# 📧 South African Spam Email Classification - Complete Pipeline

## End-to-End Data Science & Machine Learning Implementation

### 🎯 Project Overview
This notebook implements a complete spam classification system using:
- **Logistic Regression**
- **Naive Bayes (MultinomialNB)**
- **Support Vector Machine (SVM)**
- **Ensemble Model (Voting Classifier)**

### 📋 Pipeline Stages
1. Data Loading & Exploration
2. Exploratory Data Analysis (EDA)
3. Data Cleaning & Preprocessing
4. Feature Engineering (TF-IDF)
5. Model Training (LR, NB, SVM)
6. Ensemble Model Creation
7. Model Evaluation & Comparison
8. Production Predictions

---
## 📦 1. Setup & Installations

In [ ]:
# Install required packages
!pip install -q wordcloud imbalanced-learn

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter
import re
import string
from datetime import datetime

# Text processing
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from wordcloud import WordCloud

# ML preprocessing
from sklearn.model_selection import (
    train_test_split, cross_val_score, 
    StratifiedKFold, GridSearchCV
)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder

# ML models - Our three core algorithms
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC

# Ensemble model
from sklearn.ensemble import VotingClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, 
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score
)

# Settings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Download NLTK resources
print("Downloading NLTK resources...")
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("\n✅ All libraries imported successfully!")
print(f"📅 Notebook initialized: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 📊 2. Data Loading & Initial Exploration

In [ ]:
# Upload your dataset
from google.colab import files
print("📤 Please upload your spam dataset CSV file...\n")
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"\n✅ File uploaded successfully: {filename}")

In [ ]:
# Load the dataset
df = pd.read_csv(filename)

print("📊 DATASET OVERVIEW")
print("=" * 80)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n" + "=" * 80)

# Display first few rows
df.head(10)

In [ ]:
# Detailed dataset information
print("📋 DATASET INFORMATION")
print("=" * 80)
df.info()

print("\n\n🔍 MISSING VALUES CHECK")
print("=" * 80)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_df)

print("\n\n📊 DUPLICATE ROWS")
print("=" * 80)
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates:,} ({duplicates/len(df)*100:.2f}%)")

---
## 🔍 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
print("🎯 CLASS DISTRIBUTION")
print("=" * 80)
class_counts = df['label'].value_counts()
class_pct = df['label'].value_counts(normalize=True) * 100

distribution_df = pd.DataFrame({
    'Count': class_counts,
    'Percentage': class_pct
})
print(distribution_df)
print(f"\nClass Balance Ratio (spam:ham): {class_counts['spam']/class_counts['ham']:.2f}:1")

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=df, x='label', palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Class Distribution - Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%d')

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Distribution - Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Text length analysis
print("📏 TEXT LENGTH ANALYSIS")
print("=" * 80)

# Combine subject and body for full text analysis
df['full_text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')
df['text_length'] = df['full_text'].str.len()
df['word_count'] = df['full_text'].str.split().str.len()

# Statistics by class
length_stats = df.groupby('label')[['text_length', 'word_count']].describe()
print(length_stats)

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Character length distribution
for label in ['ham', 'spam']:
    data = df[df['label'] == label]['text_length']
    axes[0, 0].hist(data, bins=50, alpha=0.6, label=label, 
                    color='#2ecc71' if label == 'ham' else '#e74c3c')
axes[0, 0].set_title('Character Length Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Character Count')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Word count distribution
for label in ['ham', 'spam']:
    data = df[df['label'] == label]['word_count']
    axes[0, 1].hist(data, bins=50, alpha=0.6, label=label,
                    color='#2ecc71' if label == 'ham' else '#e74c3c')
axes[0, 1].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Word Count')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Box plots
df.boxplot(column='text_length', by='label', ax=axes[1, 0], 
           patch_artist=True, grid=False)
axes[1, 0].set_title('Character Length by Class', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Label')
axes[1, 0].set_ylabel('Character Count')
plt.sca(axes[1, 0])
plt.xticks([1, 2], ['ham', 'spam'])

df.boxplot(column='word_count', by='label', ax=axes[1, 1],
           patch_artist=True, grid=False)
axes[1, 1].set_title('Word Count by Class', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Label')
axes[1, 1].set_ylabel('Word Count')
plt.sca(axes[1, 1])
plt.xticks([1, 2], ['ham', 'spam'])

plt.tight_layout()
plt.show()

In [ ]:
# Word clouds for spam and ham
print("☁️ GENERATING WORD CLOUDS...")
print("=" * 80)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Ham wordcloud
ham_text = ' '.join(df[df['label'] == 'ham']['full_text'].values)
wordcloud_ham = WordCloud(width=800, height=400, 
                          background_color='white',
                          colormap='Greens',
                          max_words=100).generate(ham_text)
axes[0].imshow(wordcloud_ham, interpolation='bilinear')
axes[0].set_title('HAM (Legitimate) Emails - Word Cloud', 
                  fontsize=16, fontweight='bold', pad=20)
axes[0].axis('off')

# Spam wordcloud
spam_text = ' '.join(df[df['label'] == 'spam']['full_text'].values)
wordcloud_spam = WordCloud(width=800, height=400,
                           background_color='white',
                           colormap='Reds',
                           max_words=100).generate(spam_text)
axes[1].imshow(wordcloud_spam, interpolation='bilinear')
axes[1].set_title('SPAM Emails - Word Cloud', 
                  fontsize=16, fontweight='bold', pad=20)
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Top common words analysis
def get_top_words(text_series, n=20):
    """Extract top N most common words"""
    words = ' '.join(text_series).lower().split()
    # Remove common stopwords and punctuation
    stop_words = set(stopwords.words('english'))
    words = [w.strip(string.punctuation) for w in words 
             if w.lower() not in stop_words and len(w) > 2]
    return Counter(words).most_common(n)

print("🔤 TOP WORDS ANALYSIS")
print("=" * 80)

# Get top words for each class
ham_words = get_top_words(df[df['label'] == 'ham']['full_text'], 15)
spam_words = get_top_words(df[df['label'] == 'spam']['full_text'], 15)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ham top words
ham_df = pd.DataFrame(ham_words, columns=['Word', 'Count'])
axes[0].barh(ham_df['Word'], ham_df['Count'], color='#2ecc71')
axes[0].set_title('Top 15 Words in HAM Emails', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Frequency')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Spam top words
spam_df = pd.DataFrame(spam_words, columns=['Word', 'Count'])
axes[1].barh(spam_df['Word'], spam_df['Count'], color='#e74c3c')
axes[1].set_title('Top 15 Words in SPAM Emails', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Frequency')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 HAM Top Words:")
print(ham_df.to_string(index=False))
print("\n📊 SPAM Top Words:")
print(spam_df.to_string(index=False))

---
## 🧹 4. Data Cleaning & Preprocessing

In [ ]:
# Text preprocessing function
def preprocess_text(text):
    """
    Comprehensive text preprocessing pipeline
    """
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove phone numbers (SA format)
    text = re.sub(r'\b\d{3}[\s-]?\d{3}[\s-]?\d{4}\b', '', text)
    text = re.sub(r'\b0\d{9}\b', '', text)
    
    # Remove currency symbols and amounts
    text = re.sub(r'[R$£€]\s?\d+[,\d]*', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return ' '.join(words)

print("🔧 Text preprocessing function defined!")
print("\nPreprocessing steps:")
print("  1. Convert to lowercase")
print("  2. Remove URLs, emails, phone numbers")
print("  3. Remove currency and numbers")
print("  4. Remove punctuation")
print("  5. Remove stopwords")
print("  6. Lemmatization")

In [ ]:
# Apply preprocessing
print("🔄 Applying text preprocessing...")
print("=" * 80)

# Create a copy for processing
df_clean = df.copy()

# Handle missing values
df_clean['subject'] = df_clean['subject'].fillna('')
df_clean['body'] = df_clean['body'].fillna('')

# Combine subject and body
df_clean['text'] = df_clean['subject'] + ' ' + df_clean['body']

# Apply preprocessing
print("Processing text... (this may take a minute)")
df_clean['processed_text'] = df_clean['text'].apply(preprocess_text)

# Remove empty texts after preprocessing
df_clean = df_clean[df_clean['processed_text'].str.len() > 0]

print(f"\n✅ Preprocessing complete!")
print(f"Final dataset shape: {df_clean.shape}")
print(f"Rows removed due to empty text: {len(df) - len(df_clean)}")

# Show examples
print("\n📝 PREPROCESSING EXAMPLES")
print("=" * 80)
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"Original: {df_clean.iloc[i]['text'][:150]}...")
    print(f"Processed: {df_clean.iloc[i]['processed_text'][:150]}...")
    print(f"Label: {df_clean.iloc[i]['label']}")

---
## ⚙️ 5. Feature Engineering with TF-IDF

In [ ]:
# Prepare features and labels
print("🎯 PREPARING FEATURES AND LABELS")
print("=" * 80)

# Features (X) and Labels (y)
X = df_clean['processed_text']
y = df_clean['label']

# Encode labels (spam=1, ham=0)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Total samples: {len(X):,}")
print(f"Features shape: {X.shape}")
print(f"Labels shape: {y_encoded.shape}")
print(f"\nLabel encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

In [ ]:
# Train-test split
print("\n📊 SPLITTING DATA")
print("=" * 80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print(f"Training set size: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing set size: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")

# Check class distribution in splits
print("\nClass distribution in training set:")
train_dist = pd.Series(y_train).value_counts()
print(f"  Ham (0): {train_dist[0]:,} ({train_dist[0]/len(y_train)*100:.1f}%)")
print(f"  Spam (1): {train_dist[1]:,} ({train_dist[1]/len(y_train)*100:.1f}%)")

print("\nClass distribution in testing set:")
test_dist = pd.Series(y_test).value_counts()
print(f"  Ham (0): {test_dist[0]:,} ({test_dist[0]/len(y_test)*100:.1f}%)")
print(f"  Spam (1): {test_dist[1]:,} ({test_dist[1]/len(y_test)*100:.1f}%)")

In [ ]:
# TF-IDF Vectorization
print("\n🔢 TF-IDF VECTORIZATION")
print("=" * 80)

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,        # Limit to top 5000 features
    min_df=2,                 # Ignore terms that appear in less than 2 documents
    max_df=0.95,              # Ignore terms that appear in more than 95% of documents
    ngram_range=(1, 2),       # Use unigrams and bigrams
    sublinear_tf=True         # Apply sublinear tf scaling
)

# Fit and transform training data
print("Fitting TF-IDF on training data...")
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform test data
print("Transforming test data...")
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"\n✅ TF-IDF vectorization complete!")
print(f"Training features shape: {X_train_tfidf.shape}")
print(f"Testing features shape: {X_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_):,}")
print(f"Feature matrix sparsity: {(1.0 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]))*100:.2f}%")

# Show top features
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\n📝 Sample features: {list(feature_names[:20])}")

---
## 🤖 6. Model Training - Individual Models

### 6.1 Logistic Regression

In [ ]:
print("🔵 TRAINING LOGISTIC REGRESSION")
print("=" * 80)

# Initialize and train
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    C=1.0,
    solver='liblinear'
)

print("Training Logistic Regression...")
lr_model.fit(X_train_tfidf, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_tfidf)
y_pred_proba_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]

# Evaluation
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_pred_proba_lr)

print(f"\n✅ Logistic Regression Results:")
print(f"  Accuracy:  {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")
print(f"  Precision: {lr_precision:.4f}")
print(f"  Recall:    {lr_recall:.4f}")
print(f"  F1-Score:  {lr_f1:.4f}")
print(f"  AUC-ROC:   {lr_auc:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_lr, 
                          target_names=['Ham', 'Spam']))

### 6.2 Naive Bayes

In [ ]:
print("🟢 TRAINING NAIVE BAYES")
print("=" * 80)

# Initialize and train
nb_model = MultinomialNB(alpha=1.0)

print("Training Naive Bayes...")
nb_model.fit(X_train_tfidf, y_train)

# Predictions
y_pred_nb = nb_model.predict(X_test_tfidf)
y_pred_proba_nb = nb_model.predict_proba(X_test_tfidf)[:, 1]

# Evaluation
nb_accuracy = accuracy_score(y_test, y_pred_nb)
nb_precision = precision_score(y_test, y_pred_nb)
nb_recall = recall_score(y_test, y_pred_nb)
nb_f1 = f1_score(y_test, y_pred_nb)
nb_auc = roc_auc_score(y_test, y_pred_proba_nb)

print(f"\n✅ Naive Bayes Results:")
print(f"  Accuracy:  {nb_accuracy:.4f} ({nb_accuracy*100:.2f}%)")
print(f"  Precision: {nb_precision:.4f}")
print(f"  Recall:    {nb_recall:.4f}")
print(f"  F1-Score:  {nb_f1:.4f}")
print(f"  AUC-ROC:   {nb_auc:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_nb, 
                          target_names=['Ham', 'Spam']))

### 6.3 Support Vector Machine (SVM)

In [ ]:
print("🔴 TRAINING SUPPORT VECTOR MACHINE")
print("=" * 80)

# Initialize and train (using LinearSVC for better performance on large datasets)
svm_model = LinearSVC(
    C=1.0,
    max_iter=1000,
    random_state=RANDOM_STATE
)

print("Training SVM... (this may take a few moments)")
svm_model.fit(X_train_tfidf, y_train)

# Predictions
y_pred_svm = svm_model.predict(X_test_tfidf)
# Note: LinearSVC doesn't have predict_proba, so we use decision_function
y_pred_decision_svm = svm_model.decision_function(X_test_tfidf)

# Evaluation
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)
svm_auc = roc_auc_score(y_test, y_pred_decision_svm)

print(f"\n✅ SVM Results:")
print(f"  Accuracy:  {svm_accuracy:.4f} ({svm_accuracy*100:.2f}%)")
print(f"  Precision: {svm_precision:.4f}")
print(f"  Recall:    {svm_recall:.4f}")
print(f"  F1-Score:  {svm_f1:.4f}")
print(f"  AUC-ROC:   {svm_auc:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_svm, 
                          target_names=['Ham', 'Spam']))

---
## 🎯 7. Ensemble Model - Voting Classifier

In [ ]:
print("🌟 CREATING ENSEMBLE MODEL")
print("=" * 80)

# Create ensemble with soft voting
# Note: We need to use SVC with probability=True for soft voting
# For large datasets, we'll use hard voting with LinearSVC

# Retrain models for ensemble
lr_ensemble = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    C=1.0,
    solver='liblinear'
)

nb_ensemble = MultinomialNB(alpha=1.0)

svm_ensemble = LinearSVC(
    C=1.0,
    max_iter=1000,
    random_state=RANDOM_STATE
)

# Create voting classifier with hard voting
ensemble_model = VotingClassifier(
    estimators=[
        ('lr', lr_ensemble),
        ('nb', nb_ensemble),
        ('svm', svm_ensemble)
    ],
    voting='hard',  # Using hard voting for LinearSVC compatibility
    n_jobs=-1
)

print("Training Ensemble Model...")
print("  ├─ Logistic Regression")
print("  ├─ Naive Bayes")
print("  └─ Support Vector Machine")
print("\nVoting Strategy: Hard Voting (Majority Vote)\n")

ensemble_model.fit(X_train_tfidf, y_train)

# Predictions
y_pred_ensemble = ensemble_model.predict(X_test_tfidf)

# Evaluation
ensemble_accuracy = accuracy_score(y_test, y_pred_ensemble)
ensemble_precision = precision_score(y_test, y_pred_ensemble)
ensemble_recall = recall_score(y_test, y_pred_ensemble)
ensemble_f1 = f1_score(y_test, y_pred_ensemble)

print(f"\n✅ Ensemble Model Results:")
print(f"  Accuracy:  {ensemble_accuracy:.4f} ({ensemble_accuracy*100:.2f}%)")
print(f"  Precision: {ensemble_precision:.4f}")
print(f"  Recall:    {ensemble_recall:.4f}")
print(f"  F1-Score:  {ensemble_f1:.4f}")

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_ensemble, 
                          target_names=['Ham', 'Spam']))

---
## 📊 8. Model Comparison & Evaluation

In [ ]:
# Compile all results
print("📈 COMPREHENSIVE MODEL COMPARISON")
print("=" * 80)

results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'SVM', 'Ensemble'],
    'Accuracy': [lr_accuracy, nb_accuracy, svm_accuracy, ensemble_accuracy],
    'Precision': [lr_precision, nb_precision, svm_precision, ensemble_precision],
    'Recall': [lr_recall, nb_recall, svm_recall, ensemble_recall],
    'F1-Score': [lr_f1, nb_f1, svm_f1, ensemble_f1],
    'AUC-ROC': [lr_auc, nb_auc, svm_auc, np.nan]  # Ensemble doesn't have AUC with hard voting
})

# Format for better display
results_display = results_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']:
    results_display[col] = results_display[col].apply(lambda x: f"{x:.4f}" if not pd.isna(x) else "N/A")

print(results_display.to_string(index=False))

# Find best model
best_model_idx = results_df['F1-Score'].idxmax()
best_model = results_df.loc[best_model_idx, 'Model']
best_f1 = results_df.loc[best_model_idx, 'F1-Score']

print(f"\n🏆 Best Model: {best_model} (F1-Score: {best_f1:.4f})")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for idx, (ax, metric) in enumerate(zip(axes.flat, metrics)):
    data = results_df[metric]
    bars = ax.bar(results_df['Model'], data, color=colors)
    ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12)
    ax.set_ylim([0, 1.05])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Rotate x-axis labels
    ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

models_data = [
    ('Logistic Regression', y_pred_lr),
    ('Naive Bayes', y_pred_nb),
    ('SVM', y_pred_svm),
    ('Ensemble', y_pred_ensemble)
]

for ax, (model_name, y_pred) in zip(axes.flat, models_data):
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Ham', 'Spam'],
                yticklabels=['Ham', 'Spam'],
                cbar=True, ax=ax, annot_kws={'size': 14})
    
    ax.set_title(f'{model_name}\nConfusion Matrix', fontsize=14, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    
    # Add accuracy to title
    accuracy = (cm[0, 0] + cm[1, 1]) / cm.sum()
    ax.text(1, -0.15, f'Accuracy: {accuracy:.4f}', 
            ha='center', va='top', transform=ax.transAxes,
            fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves (for models with probability estimates)
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Logistic Regression ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_auc:.3f})', 
        linewidth=2, color='#3498db')

# Naive Bayes ROC
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_pred_proba_nb)
ax.plot(fpr_nb, tpr_nb, label=f'Naive Bayes (AUC = {nb_auc:.3f})', 
        linewidth=2, color='#2ecc71')

# SVM ROC (using decision function)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_pred_decision_svm)
ax.plot(fpr_svm, tpr_svm, label=f'SVM (AUC = {svm_auc:.3f})', 
        linewidth=2, color='#e74c3c')

# Diagonal line (random classifier)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Logistic Regression
print("🔍 TOP PREDICTIVE FEATURES")
print("=" * 80)

# Get feature coefficients from Logistic Regression
feature_names = tfidf_vectorizer.get_feature_names_out()
coefficients = lr_model.coef_[0]

# Top spam indicators (positive coefficients)
top_spam_indices = np.argsort(coefficients)[-20:][::-1]
top_spam_features = [(feature_names[i], coefficients[i]) for i in top_spam_indices]

# Top ham indicators (negative coefficients)
top_ham_indices = np.argsort(coefficients)[:20]
top_ham_features = [(feature_names[i], coefficients[i]) for i in top_ham_indices]

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Spam indicators
spam_words = [f[0] for f in top_spam_features[:15]]
spam_scores = [f[1] for f in top_spam_features[:15]]
axes[0].barh(range(len(spam_words)), spam_scores, color='#e74c3c')
axes[0].set_yticks(range(len(spam_words)))
axes[0].set_yticklabels(spam_words)
axes[0].set_xlabel('Coefficient (Importance)', fontsize=12)
axes[0].set_title('Top 15 SPAM Indicators', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Ham indicators
ham_words = [f[0] for f in top_ham_features[:15]]
ham_scores = [abs(f[1]) for f in top_ham_features[:15]]
axes[1].barh(range(len(ham_words)), ham_scores, color='#2ecc71')
axes[1].set_yticks(range(len(ham_words)))
axes[1].set_yticklabels(ham_words)
axes[1].set_xlabel('Coefficient (Importance)', fontsize=12)
axes[1].set_title('Top 15 HAM Indicators', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTop 10 SPAM indicators:")
for word, coef in top_spam_features[:10]:
    print(f"  {word:30s} {coef:>8.4f}")

print("\nTop 10 HAM indicators:")
for word, coef in top_ham_features[:10]:
    print(f"  {word:30s} {abs(coef):>8.4f}")

---
## 🚀 9. Production-Ready Predictions

In [ ]:
def predict_spam(email_text, model='ensemble'):
    """
    Predict if an email is spam or ham
    
    Parameters:
    -----------
    email_text : str
        The email text to classify
    model : str
        Which model to use: 'lr', 'nb', 'svm', or 'ensemble'
    
    Returns:
    --------
    dict : prediction results
    """
    # Preprocess the text
    processed_text = preprocess_text(email_text)
    
    # Vectorize
    text_tfidf = tfidf_vectorizer.transform([processed_text])
    
    # Select model
    model_map = {
        'lr': lr_model,
        'nb': nb_model,
        'svm': svm_model,
        'ensemble': ensemble_model
    }
    
    selected_model = model_map.get(model, ensemble_model)
    
    # Predict
    prediction = selected_model.predict(text_tfidf)[0]
    
    # Get probability if available
    if hasattr(selected_model, 'predict_proba'):
        probabilities = selected_model.predict_proba(text_tfidf)[0]
        confidence = probabilities[prediction]
    else:
        confidence = None
    
    result = {
        'prediction': 'SPAM' if prediction == 1 else 'HAM',
        'prediction_code': int(prediction),
        'confidence': confidence,
        'model_used': model,
        'original_text': email_text,
        'processed_text': processed_text
    }
    
    return result

print("✅ Prediction function created!")
print("\nUsage: predict_spam('your email text here', model='ensemble')")
print("Available models: 'lr', 'nb', 'svm', 'ensemble'")

In [ ]:
# Test with sample emails
print("🧪 TESTING PREDICTIONS ON SAMPLE EMAILS")
print("=" * 80)

test_emails = [
    "Congratulations! You've won R1,000,000! Click here to claim your prize now!",
    "Hi John, let's meet for coffee tomorrow at 3pm. Looking forward to catching up!",
    "URGENT: Your account will be suspended. Verify your details immediately at suspicious-link.com",
    "Your Amazon order #12345 has been shipped. Expected delivery: 2 days.",
    "Make R50,000 per day working from home! No experience needed! Send R500 registration fee."
]

for i, email in enumerate(test_emails, 1):
    print(f"\n{'='*80}")
    print(f"Test Email #{i}")
    print(f"{'='*80}")
    print(f"Text: {email}")
    print()
    
    # Test with all models
    for model_name in ['lr', 'nb', 'svm', 'ensemble']:
        result = predict_spam(email, model=model_name)
        
        model_labels = {
            'lr': 'Logistic Regression',
            'nb': 'Naive Bayes',
            'svm': 'SVM',
            'ensemble': 'Ensemble'
        }
        
        conf_str = f"(Confidence: {result['confidence']:.2%})" if result['confidence'] else ""
        
        emoji = "🚨" if result['prediction'] == 'SPAM' else "✅"
        print(f"  {emoji} {model_labels[model_name]:20s}: {result['prediction']} {conf_str}")

In [ ]:
# Interactive prediction widget
def interactive_predict():
    """Interactive spam prediction interface"""
    print("\n" + "="*80)
    print("📧 INTERACTIVE SPAM DETECTOR")
    print("="*80)
    print("Enter an email text to classify (or 'quit' to exit)\n")
    
    while True:
        user_input = input("\nEmail text: ").strip()
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Goodbye!")
            break
        
        if not user_input:
            print("⚠️  Please enter some text!")
            continue
        
        # Get prediction from ensemble model
        result = predict_spam(user_input, model='ensemble')
        
        print("\n" + "-"*80)
        if result['prediction'] == 'SPAM':
            print("🚨 CLASSIFICATION: SPAM")
            print("⚠️  This email appears to be spam/malicious")
        else:
            print("✅ CLASSIFICATION: HAM (Legitimate)")
            print("✓  This email appears to be legitimate")
        print("-"*80)

# Uncomment the line below to run the interactive predictor
# interactive_predict()

---
## 💾 10. Model Saving & Export

In [ ]:
import pickle
from datetime import datetime

print("💾 SAVING MODELS AND VECTORIZER")
print("=" * 80)

# Create timestamp for versioning
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save all models
models_to_save = {
    f'logistic_regression_{timestamp}.pkl': lr_model,
    f'naive_bayes_{timestamp}.pkl': nb_model,
    f'svm_{timestamp}.pkl': svm_model,
    f'ensemble_{timestamp}.pkl': ensemble_model,
    f'tfidf_vectorizer_{timestamp}.pkl': tfidf_vectorizer
}

for filename, model in models_to_save.items():
    with open(filename, 'wb') as file:
        pickle.dump(model, file)
    print(f"✅ Saved: {filename}")

print("\n📦 All models and vectorizer saved successfully!")
print("\nTo load a model later:")
print("  with open('model_name.pkl', 'rb') as file:")
print("      model = pickle.load(file)")

In [ ]:
# Export results summary
print("\n📄 EXPORTING RESULTS SUMMARY")
print("=" * 80)

# Create comprehensive results dataframe
export_results = results_df.copy()
export_results['Training_Samples'] = len(X_train)
export_results['Testing_Samples'] = len(X_test)
export_results['Feature_Count'] = X_train_tfidf.shape[1]
export_results['Timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Save to CSV
results_filename = f'model_comparison_results_{timestamp}.csv'
export_results.to_csv(results_filename, index=False)
print(f"✅ Results exported to: {results_filename}")

# Download files
from google.colab import files
print("\n📥 Downloading files...")
files.download(results_filename)

print("\n✅ Export complete!")

---
## 📝 Summary & Conclusions

### Key Findings:
1. **Dataset**: Successfully processed 100,000 South African spam emails
2. **Models Trained**:
   - Logistic Regression
   - Naive Bayes (MultinomialNB)
   - Support Vector Machine (LinearSVC)
   - Ensemble Model (Voting Classifier)

### Best Performing Model:
Review the comparison metrics above to determine the best model for your specific use case.

### Production Deployment:
- All models saved with timestamps
- TF-IDF vectorizer saved for consistency
- Prediction function ready for integration
- Results exported for documentation

### Next Steps:
1. Deploy the best model to production
2. Monitor model performance over time
3. Retrain periodically with new data
4. Consider implementing active learning for continuous improvement

---

**Notebook Created**: End-to-End Spam Classification Pipeline

**Author**: Data Science Team

**Date**: {datetime.now().strftime('%Y-%m-%d')}

---